#### Inisialisasi

In [1]:
import findspark
findspark.init()  # Menghubungkan VS Code ke Apache Spark lokal

import mlflow
import mlflow.spark
import math
from pyspark.ml.clustering import GaussianMixture
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("GMM_Reliability_Analysis")

spark = SparkSession.builder \
    .appName("Training_GMM_ASEAN_Pipeline") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

c:\Users\marit\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/05/21 14:55:13 INFO mlflow.tracking.fluent: Experiment with name 'GMM_Reliability_Analysis' does not exist. Creating a new experiment.


#### Load Data Bersih

In [2]:
HDFS_INPUT = "hdfs://localhost:9000/Project_akhir/data_bersih_asean"

print(f"Membaca data bersih dari HDFS: {HDFS_INPUT}")
df = spark.read.parquet(HDFS_INPUT)

df = df.cache()
total_records = df.count()
print(f"Total baris data: {total_records:,}")

Membaca data bersih dari HDFS: hdfs://localhost:9000/Project_akhir/data_bersih_asean
Total baris data: 3,857,065


#### Looping Eksperimen GMM & Logging Otomatis ke MLflow

In [3]:
# Definisi variabel
best_log_likelihood = -float('inf')
best_k = 2
best_model = None
best_predictions = None
best_bic = None

hasil_metrik_gmm = []

print("Pencarian jumlah cluster optimal menggunakan GMM")
for k in range(2, 6):
    with mlflow.start_run(run_name=f"GMM_K_{k}"):
        # penggunaan kolom "reliability_metrics" sebagai fitur untuk clustering gmm
        gmm = GaussianMixture(
            k=k, 
            featuresCol="reliability_metrics", 
            predictionCol="gmm_cluster", 
            seed=42
        )
        
        # training model
        model = gmm.fit(df)
        predictions_full = model.transform(df)
        
        # Ekstraksi metrik Log Likelihood dari summary model
        summary = model.summary
        ll = summary.logLikelihood
        
        # Perhitungan BIC = k_params * ln(n) - 2 * log_likelihood
        num_features = df.select("reliability_metrics").first()[0].size
        k_params = (k * num_features) + (k - 1) 
        bic = k_params * math.log(total_records) - 2 * ll
        
        # Menulis parameter dan metrik secara real-time ke database mlflow.db
        mlflow.log_param("k", k)
        mlflow.log_metric("log_likelihood", ll)
        mlflow.log_metric("BIC", bic)
        
        # Simpan metrik dalam list 
        hasil_metrik_gmm.append({
            "k": k,
            "log_likelihood": ll,
            "bic": bic,
            "model": model,
            "predictions": predictions_full
        })
        
        print(f"Iterasi K={k} Selesai | Log Likelihood: {ll:.4f} | BIC: {bic:.4f}")
        
        # Penentuan Model Terbaik: Mencari Log Likelihood yang paling mendekati 0 (terbesar)
        if ll > best_log_likelihood:
            best_log_likelihood = ll
            best_bic = bic  # Kunci nilai BIC dari model terbaik
            best_k = k
            best_model = model
            best_predictions = predictions_full

# Ringkasan hasil pencarian model terbaik
print("\nRingkasan Pencarian Model GMM Terbaik")
print(f"{'K':>4}   {'Log Likelihood':>16}   {'BIC':>16}   {'Status':>10}")
for r in hasil_metrik_gmm:
    # Flag penanda cluster terbaik 
    flag = "TERBAIK" if r["k"] == best_k else ""
    print(f"{r['k']:>4}   {r['log_likelihood']:>16.4f}   {r['bic']:>16.2f}   {flag}")

print("-" * 60)
print(f"K Komponen Terbaik : {best_k}")
print(f"Log Likelihood     : {best_log_likelihood:.4f}")
print(f"BIC Terbaik        : {best_bic:.2f}")

Pencarian jumlah cluster optimal menggunakan GMM
Iterasi K=2 Selesai | Log Likelihood: 31120795.9038 | BIC: -62241515.9804
🏃 View run GMM_K_2 at: http://localhost:5000/#/experiments/2/runs/fe5abcc5e0904ccd8e7c6da5a156a83d
🧪 View experiment at: http://localhost:5000/#/experiments/2
Iterasi K=3 Selesai | Log Likelihood: 31869004.6215 | BIC: -63737887.9197
🏃 View run GMM_K_3 at: http://localhost:5000/#/experiments/2/runs/9b4ed0ca37104df28692d91e37aae8f4
🧪 View experiment at: http://localhost:5000/#/experiments/2
Iterasi K=4 Selesai | Log Likelihood: 32520978.3731 | BIC: -65041789.9266
🏃 View run GMM_K_4 at: http://localhost:5000/#/experiments/2/runs/0a9f6ef6b89c42d2bc50c50ffd0d8a2f
🧪 View experiment at: http://localhost:5000/#/experiments/2
Iterasi K=5 Selesai | Log Likelihood: 34098456.0446 | BIC: -68196699.7733
🏃 View run GMM_K_5 at: http://localhost:5000/#/experiments/2/runs/eded9aa7da9648ea92c369f26638a021
🧪 View experiment at: http://localhost:5000/#/experiments/2

Ringkasan Pencaria

#### Menyimpan Hasil Prediksi

In [5]:
if best_model is not None:
    HDFS_OUTPUT_GMM = "hdfs://localhost:9000/Project_akhir/hasil_gmm_reliability"

    print(f"Menyimpan hasil klasterisasi GMM ke HDFS: {HDFS_OUTPUT_GMM}")
    best_predictions.write.mode("overwrite").parquet(HDFS_OUTPUT_GMM)
    print("Data berhasil disimpan.")

    # Log model ke MLflow dalam run baru yang eksplisit
    with mlflow.start_run(run_name=f"Final_Best_GMM_K{best_k}"):
        mlflow.log_param("best_k",           best_k)
        mlflow.log_metric("log_likelihood",  best_log_likelihood)
        mlflow.log_metric("bic",             best_bic)
    print(f"Model GMM K={best_k} berhasil di-log ke MLflow.")

Menyimpan hasil klasterisasi GMM ke HDFS: hdfs://localhost:9000/Project_akhir/hasil_gmm_reliability
Data berhasil disimpan.
🏃 View run Final_Best_GMM_K5 at: http://localhost:5000/#/experiments/2/runs/8f63e7b3dc324bf69ffe0621d4d88636
🧪 View experiment at: http://localhost:5000/#/experiments/2
Model GMM K=5 berhasil di-log ke MLflow.


#### Profiling

In [6]:
path_output_viz_gmm = "hdfs://localhost:9000/Project_akhir/visualisasi_asean/gmm_cluster_profile"

# 1. PROFILING STATISTIK UTAMA (Menentukan arti tingkat keandalan setiap klaster)
# Mengelompokkan berdasarkan klaster untuk melihat rata-rata metrik mentah sebelum scaling
gmm_profile_stat = best_predictions.groupBy("gmm_cluster") \
    .agg(
        F.avg("SAM").alias("avg_sam"),
        F.avg("data_age_days").alias("avg_days_old"),
        F.count("*").alias("tower_count")
    ).orderBy(F.col("avg_sam").desc())

gmm_profile_stat.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{path_output_viz_gmm}/stats_utama_gmm")

# 2. PROFILING DISTRIBUSI OPERATOR DAN NEGARA (Mendeteksi akurasi pembaruan tiap provider)
# Mengetahui penyebaran kualitas data menara milik operator di tiap-tiap negara ASEAN
op_reliability = best_predictions.groupBy("Country", "Network", "gmm_cluster") \
    .agg(
        F.count("*").alias("tower_count")
    ).orderBy("Country", "Network", "gmm_cluster")

op_reliability.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{path_output_viz_gmm}/distribusi_keandalan_operator")

print("Selesai")

Selesai
